# ChemBreak Adaptive Jailbreak v3

GitHub-to-Colab Enterprise runner for `https://github.com/Jollychuks/ChemBreak`.

Project folder: `ChemBreak_Adaptive_Jailbreak_v3`

**Run TEST first.** V3 uses a fresh output namespace. It retains the V2 storage fix and adds clean attacker/judge history separation, explicit attacker progress, structured Gemini chemistry judging, and improved GPT-OSS rate-limit pacing.


## 1. Clone or refresh the ChemBreak GitHub repository

This cell changes only the ephemeral Colab Enterprise runtime. It never pushes to GitHub.


In [ ]:
from pathlib import Path
import os, subprocess

REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH = "main"
PROJECT_SUBDIR = "ChemBreak_Adaptive_Jailbreak_v3"
WORK_ROOT = Path("/content") if Path("/content").exists() else (Path.home() / "chembreak_colab")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORK_ROOT / "ChemBreak"

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

PROJECT_DIR = REPO_DIR / PROJECT_SUBDIR
assert PROJECT_DIR.exists(), f"Project folder not found: {PROJECT_DIR}. Upload {PROJECT_SUBDIR} to GitHub first."
os.chdir(PROJECT_DIR)
print("Repository:", REPO_DIR)
print("Project directory:", PROJECT_DIR)
print("Git commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 2. Route model caches to the large `/content` disk

This must run before model downloads. The prior run filled the small system disk because Hugging Face defaulted to `/root/.cache/huggingface`. V3 explicitly uses `/content/hf_cache` and can remove only the obsolete Hugging Face cache on the ephemeral system disk.


In [ ]:
from pathlib import Path
import os, shutil

HF_CACHE_DIR = Path("/content/hf_cache") if Path("/content").exists() else (Path.home()/".cache"/"huggingface")
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
(Path(HF_CACHE_DIR).parent / "tmp").mkdir(parents=True, exist_ok=True)
(Path(HF_CACHE_DIR) / "xet").mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
os.environ["HF_XET_CACHE"] = str(HF_CACHE_DIR / "xet")
os.environ["TORCH_HOME"] = str(HF_CACHE_DIR.parent / "torch_cache")
os.environ["TMPDIR"] = str(HF_CACHE_DIR.parent / "tmp")

# Safe in this ephemeral Colab Enterprise runtime. It does not touch GCS or GitHub.
legacy = Path.home()/".cache"/"huggingface"
if legacy.exists() and legacy.resolve() != HF_CACHE_DIR.resolve():
    print("Removing obsolete system-disk HF cache:", legacy)
    shutil.rmtree(legacy)

for label, path in [("system", Path("/")), ("content", Path("/content") if Path("/content").exists() else HF_CACHE_DIR)]:
    u = shutil.disk_usage(path)
    print(f"{label}: {u.used/2**30:.1f} GB used / {u.total/2**30:.1f} GB total | {u.free/2**30:.1f} GB free")
print("HF cache:", HF_CACHE_DIR)


## 3. Install dependencies

The CUDA-enabled PyTorch already in the runtime is retained. `--no-cache-dir` prevents pip from filling the system disk with package-download caches.


In [ ]:
!python -m pip install -q --no-cache-dir -r requirements.txt
!python -m pip install -q --no-cache-dir -e .
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


## 4. Configure the run and model roles

Keep `RUN_ID` stable only when resuming the same V3 run. V3 uses its own GCS namespace and does not mix results with V2 or earlier versions.


In [ ]:
GCP_PROJECT = "rs-foundsecft-mghasemi"
GCS_BUCKET = "rs-foundsecft-mghasemi-default-1"
RUN_MODE = "test"      # test | pilot | production
RUN_ID = "test_001"    # keep stable to resume this V3 run

ATTACKER_MODEL = "gemini-3.1-pro-preview"
SAFETY_JUDGE_MODEL = "openai/gpt-oss-120b-maas"
CHEMISTRY_JUDGE_MODEL = "gemini-2.5-pro"
ADJUDICATOR_MODEL = "meta/llama-4-maverick-17b-128e-instruct-maas"

TASK_BANK_PREFIX = "ChemBreak_V15/outputs/production/"
GCS_BASE_URI = f"gs://{GCS_BUCKET}/ChemBreak_Adaptive_Jailbreak_v3"
GCS_OUTPUT_URI = f"{GCS_BASE_URI}/{RUN_MODE}/{RUN_ID}"

print("Project:", GCP_PROJECT)
print("Mode:", RUN_MODE)
print("Run ID:", RUN_ID)
print("Output:", GCS_OUTPUT_URI)
print("HF cache:", HF_CACHE_DIR)
print("Attacker:", ATTACKER_MODEL)
print("Safety judge:", SAFETY_JUDGE_MODEL)
print("Chemistry judge:", CHEMISTRY_JUDGE_MODEL)
print("Adjudicator:", ADJUDICATOR_MODEL)


## 5. Locate the final ChemBreak task bank in Google Cloud Storage


In [ ]:
from google.cloud import storage
client = storage.Client(project=GCP_PROJECT)
blobs = list(client.list_blobs(GCS_BUCKET, prefix=TASK_BANK_PREFIX))
matches = [b for b in blobs if b.name.lower().endswith(".csv") and "final_task_bank" in b.name.lower()]
matches.sort(key=lambda b: b.updated or 0, reverse=True)
if not matches:
    raise FileNotFoundError(f"No final_task_bank CSV found under gs://{GCS_BUCKET}/{TASK_BANK_PREFIX}.")
print("Matching final task banks:")
for i, b in enumerate(matches, 1):
    print(f"{i}. gs://{GCS_BUCKET}/{b.name} | updated={b.updated}")
TASK_BANK_URI = f"gs://{GCS_BUCKET}/{matches[0].name}"
print("Selected:", TASK_BANK_URI)


## 6. Create the local runtime configuration

`configs/runtime.yaml` is ignored by Git.


In [ ]:
import subprocess
subprocess.run([
    "python", "scripts/create_runtime_config.py",
    "--template", "configs/gcp.yaml",
    "--output", "configs/runtime.yaml",
    "--project", GCP_PROJECT,
    "--run-mode", RUN_MODE,
    "--run-id", RUN_ID,
    "--task-bank-uri", TASK_BANK_URI,
    "--gcs-output-uri", GCS_OUTPUT_URI,
    "--attacker-model", ATTACKER_MODEL,
    "--safety-judge-model", SAFETY_JUDGE_MODEL,
    "--chemistry-judge-model", CHEMISTRY_JUDGE_MODEL,
    "--adjudicator-model", ADJUDICATOR_MODEL,
    "--hf-cache-dir", str(HF_CACHE_DIR),
], check=True)
CONFIG = "configs/runtime.yaml"
print(Path(CONFIG).read_text()[:8000])


## 7. Preflight

V3 preflight verifies the cache path and free disk space in addition to GPU, task-bank schema, target repositories, RDKit, and harmless cloud-model connectivity. It refuses to start if the model-cache disk has less than 80 GB free or the system disk has less than 5 GB free.


In [ ]:
!python scripts/preflight.py --config $CONFIG


## 8. Prepare frozen attack assets

Test uses 8 task-bank prompts, pilot uses 40, and production uses all 500. Attack assets are checkpointed to the V3 GCS namespace. Gemini 3.1 Pro Preview is the attacker used here to create C1 attempts, the C2 fixed sequence, and the C3 route graph.


In [ ]:
!python scripts/run.py prepare --config $CONFIG


### V3 attacker visibility during execution

C0 has no attacker. C1 and C2 execute frozen assets created during PREPARE. C3 calls Gemini 3.1 Pro Preview live after target responses. V3 prints `[C3 ATTACKER]` status lines for those decisions while keeping generated attack text out of the console. Judge feedback is OFF in the primary configuration.


## 9. Run ChemDFM

ChemDFM is loaded once and reused. Judge/network retries are automatic. A target-load failure aborts this cell immediately instead of printing the same failure 32 times.


In [ ]:
!python scripts/run.py execute --config $CONFIG --target chemdfm


## 10. Run ChemLLM


In [ ]:
!python scripts/run.py execute --config $CONFIG --target chemllm


## 11. Run LlaSMol


In [ ]:
!python scripts/run.py execute --config $CONFIG --target llasmol


## 12. Rebuild aggregate metrics

Metrics include alignment-breach rate, effective-chemical-breach rate, query efficiency, judge disagreement, adjudication rate, and verifier contradiction rate.


In [ ]:
!python scripts/run.py metrics --config $CONFIG


## 13. Resume behavior

If the runtime stops, reconnect, rerun Sections 1 through 7 using the same V3 `RUN_MODE` and `RUN_ID`, then rerun the interrupted stage. GCS checkpoints are restored and completed task-target-condition units are skipped. Do not reuse a V3 `RUN_ID` for an intentionally fresh run.
